In [ ]:
import os
import sys
import time
import json
import getpass
import subprocess
import warnings
warnings.filterwarnings("ignore")

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "transformers>=4.51.0",
    "accelerate",
    "sentencepiece",
    "safetensors",
    "pandas",
)

import torch
import pandas as pd
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

print("=" * 100)
print("STEP 1 — Hugging Face authentication")
print("=" * 100)

hf_token = None
try:
    from google.colab import userdata
    try:
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
except Exception:
    pass

if not hf_token:
    hf_token = getpass.getpass("Enter your Hugging Face token: ").strip()

login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
print("HF login successful.")

In [ ]:
print("=" * 100)
print("STEP 2 — Device setup")
print("=" * 100)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print("device:", device)
print("dtype:", dtype)

model_id = "google/gemma-3-1b-it"
print("model_id:", model_id)

print("=" * 100)
print("STEP 3 — Load tokenizer and model")
print("=" * 100)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    torch_dtype=dtype,
    device_map="auto",
)

model.eval()
print("Tokenizer and model loaded successfully.")

In [ ]:
def build_chat_prompt(user_prompt: str):
    messages = [
        {"role": "user", "content": user_prompt}
    ]
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        text = f"<start_of_turn>user\n{user_prompt}<end_of_turn>\n<start_of_turn>model\n"
    return text

def generate_text(prompt, max_new_tokens=256, temperature=0.7, do_sample=True):
    chat_text = build_chat_prompt(prompt)
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=0.95 if do_sample else None,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print("=" * 100)
print("STEP 4 — Basic generation")
print("=" * 100)

prompt1 = """Explain Gemma 3 in plain English.
Then give:
1. one practical use case
2. one limitation
3. one Colab tip
Keep it concise."""
resp1 = generate_text(prompt1, max_new_tokens=220, temperature=0.7, do_sample=True)
print(resp1)

In [ ]:
print("=" * 100)
print("STEP 5 — Structured output")
print("=" * 100)

prompt2 = """
Compare local open-weight model usage vs API-hosted model usage.

Return JSON with this schema:
{
  "local": {
    "pros": ["", "", ""],
    "cons": ["", "", ""]
  },
  "api": {
    "pros": ["", "", ""],
    "cons": ["", "", ""]
  },
  "best_for": {
    "local": "",
    "api": ""
  }
}
Only output JSON.
"""
resp2 = generate_text(prompt2, max_new_tokens=300, temperature=0.2, do_sample=True)
print(resp2)

print("=" * 100)
print("STEP 6 — Prompt chaining")
print("=" * 100)

task = "Draft a 5-step checklist for evaluating whether Gemma fits an internal enterprise prototype."
resp3 = generate_text(task, max_new_tokens=250, temperature=0.6, do_sample=True)
print(resp3)

followup = f"""
Here is an initial checklist:

{resp3}

Now rewrite it for a product manager audience.
"""
resp4 = generate_text(followup, max_new_tokens=250, temperature=0.6, do_sample=True)
print(resp4)

In [6]:
print("=" * 100)
print("STEP 7 — Mini benchmark")
print("=" * 100)

prompts = [
    "Explain tokenization in two lines.",
    "Give three use cases for local LLMs.",
    "What is one downside of small local models?",
    "Explain instruction tuning in one paragraph."
]

rows = []
for p in prompts:
    t0 = time.time()
    out = generate_text(p, max_new_tokens=140, temperature=0.3, do_sample=True)
    dt = time.time() - t0
    rows.append({
        "prompt": p,
        "latency_sec": round(dt, 2),
        "chars": len(out),
        "preview": out[:160].replace("\n", " ")
    })

df = pd.DataFrame(rows)
print(df)

print("=" * 100)
print("STEP 8 — Deterministic summarization")
print("=" * 100)

long_text = """
Gemma is a family of lightweight open models from Google. In practical usage, teams often evaluate
trade-offs among local deployment cost, latency, privacy, controllability, and raw capability.
Smaller models can be easier to deploy, but they may struggle more on complex reasoning or domain-specific tasks.
"""

summary_prompt = f"""
Summarize the following in exactly 4 bullet points:

{long_text}
"""
summary = generate_text(summary_prompt, max_new_tokens=180, do_sample=False)
print(summary)

print("=" * 100)
print("STEP 9 — Save outputs")
print("=" * 100)

report = {
    "model_id": model_id,
    "device": str(model.device),
    "basic_generation": resp1,
    "structured_output": resp2,
    "chain_step_1": resp3,
    "chain_step_2": resp4,
    "summary": summary,
    "benchmark": rows,
}

with open("gemma3_1b_text_tutorial_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Saved gemma3_1b_text_tutorial_report.json")
print("Tutorial complete.")

STEP 1 — Hugging Face authentication
Enter your Hugging Face token: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login successful.
STEP 2 — Device setup
device: cuda
dtype: torch.bfloat16
model_id: google/gemma-3-1b-it
STEP 3 — Load tokenizer and model


tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Tokenizer and model loaded successfully.
STEP 4 — Basic generation
Okay, here’s a breakdown of Gemma 3, explained simply, followed by your requested elements:

**Gemma 3 Explained:**

Gemma 3 is a new AI model created by the Gemma team at Google DeepMind. It’s designed to be a powerful language model – meaning it can understand and generate text – and it’s particularly good at following instructions and being creative. Think of it as a really smart, helpful assistant that can write different kinds of text formats, translate languages, and answer your questions in a comprehensive way. It’s open-weights, which means you can use it freely (subject to the license).

**1. Practical Use Case:**

*   **Drafting Emails:** Quickly generate drafts for emails based on a brief description of the desired message – like a customer service response or a proposal.

**2. Limitation:**

*   **Can Hallucinate:** It can sometimes confidently provide incorrect or fabricated information – it doesn’t "know" 

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


                                         prompt  latency_sec  chars  \
0            Explain tokenization in two lines.         9.78    487   
1          Give three use cases for local LLMs.         9.81    577   
2   What is one downside of small local models?         9.59    642   
3  Explain instruction tuning in one paragraph.         9.62    638   

                                             preview  
0  Tokenization is the process of breaking down a...  
1  Okay, here are three use cases for local LLMs,...  
2  Okay, here’s one significant downside of small...  
3  Instruction tuning is a technique in large lan...  
STEP 8 — Deterministic summarization
Here's a 4-bullet point summary of the provided text:

*   **Gemma is a family of lightweight AI models.**
*   **Deploying these models is often a trade-off between cost, speed, privacy, and ability to perform tasks.**
*   **Smaller models are generally easier to deploy but may have limitations with complex reasoning or specialize